# 01 - Extraccion de Datos

**Proyecto:** Pochoclo Predict
**Materia:** Web Mining - Trabajo Practico Final

## Objetivo de este notebook

Construir el dataset crudo de peliculas que se utilizara en todo el proyecto,
consultando la API publica de [The Movie Database (TMDB)](https://www.themoviedb.org/documentation/api).

La variable objetivo (target) del proyecto es **`nota_promedio`**: el promedio
de calificacion de los usuarios de TMDB para cada pelicula, en una escala de
1 a 10. El objetivo final del TP es predecir esta nota a partir de atributos
disponibles *antes o al momento del estreno* (presupuesto, duracion, genero,
elenco, sinopsis, etc.), es decir, un problema de **regresion supervisada**.

## Metodologia de extraccion

TMDB no permite descargar el catalogo completo en un solo llamado: hay que
recorrer un endpoint de busqueda/descubrimiento paginado y luego pedir el
detalle de cada pelicula por separado. Concretamente:

1. **Descubrimiento (`/discover/movie`)**: se recorren paginas ordenadas por
   `vote_count.desc` (cantidad de votos descendente). Ordenar por cantidad de
   votos -en lugar de por popularidad o por nota- prioriza peliculas cuya
   `nota_promedio` es estadisticamente mas confiable: una pelicula con 2 votos
   y nota 10 es ruido, no señal. Se aplica ademas un piso minimo de votos
   (`MIN_VOTE_COUNT`) para descartar directamente esos casos extremos.
2. **Detalle (`/movie/{id}`)**: por cada id descubierto se pide el detalle
   completo en **una sola llamada**, combinando con `append_to_response` la
   informacion de reparto/equipo (`credits`), palabras clave (`keywords`) y
   traducciones (`translations`) — esta ultima para poder recuperar tambien
   la sinopsis en ingles, que se usara mas adelante para el analisis de
   sentimiento (NB03).

Toda la logica de acceso a la API vive en `src/tmdb_client.py` para que este
notebook se enfoque en el *que* y no en el *como*.


In [1]:
import sys
from pathlib import Path

# Se agrega la raiz del proyecto al sys.path para poder importar el paquete `src`.
# Path.cwd() es la carpeta del propio notebook (Jupyter la fija como cwd por
# defecto), por lo que esto funciona sin importar en que ubicacion del disco
# este copiado el proyecto.
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import os
import pandas as pd
from tqdm.auto import tqdm

from src import paths
from src.tmdb_client import TMDBClient

pd.set_option("display.max_columns", None)


## 1. Configuracion y conexion a la API

La API Key se lee desde un archivo `.env` en la raiz del proyecto (no se
hardcodea en el codigo, para poder compartir/versionar el proyecto sin
exponer credenciales). Ver `.env.example` para el formato esperado.


In [2]:
API_KEY = os.environ.get("TMDB_API_KEY")
client = TMDBClient(api_key=API_KEY, language="es-ES")

# Parametros de la extraccion
MIN_VOTE_COUNT = 30   # piso de votos para considerar confiable la nota_promedio
MAX_PAGES = 150        # 150 paginas x 20 resultados = hasta 3000 peliculas candidatas

print("Cliente TMDB inicializado correctamente.")


Cliente TMDB inicializado correctamente.


## 2. Descubrimiento del universo de peliculas

Se obtienen los ids de las peliculas candidatas, ordenadas por cantidad de
votos descendente.


In [3]:
movie_ids = client.discover_movie_ids(min_vote_count=MIN_VOTE_COUNT, max_pages=MAX_PAGES)
movie_ids = list(dict.fromkeys(movie_ids))  # dedupe preservando el orden

print(f"Peliculas candidatas descubiertas: {len(movie_ids)}")


Peliculas candidatas descubiertas: 3000


## 3. Descarga del detalle completo por pelicula

Por cada id se descarga el detalle completo (incluyendo reparto, palabras
clave y traducciones) y se aplana a un registro tabular con
`TMDBClient.parse_movie`. Las peliculas que fallan la consulta (timeouts,
errores puntuales de la API) se descartan silenciosamente: a esta escala,
perder un puñado de registros no afecta la calidad del dataset.


In [4]:
registros = []
fallidos = 0

for movie_id in tqdm(movie_ids, desc="Descargando detalle"):
    raw = client.get_movie_full(movie_id)
    fila = TMDBClient.parse_movie(raw)
    if fila is not None:
        registros.append(fila)
    else:
        fallidos += 1

print(f"Peliculas descargadas correctamente: {len(registros)}")
print(f"Peliculas descartadas por error de consulta: {fallidos}")


Descargando detalle:   0%|          | 0/3000 [00:00<?, ?it/s]

Peliculas descargadas correctamente: 3000
Peliculas descartadas por error de consulta: 0


## 4. Construccion del dataset final

Se arma el DataFrame y se aplican filtros minimos de calidad:

- Se descartan duplicados por `id`.
- Se descartan peliculas sin `nota_promedio` (sin votos suficientes para
  tener una nota calculada) o sin `duracion_min` (metadata incompleta).

No se hace aca ninguna transformacion de features (eso corresponde al NB03):
este notebook entrega el dataset **crudo**, tal como llega de la API.


In [5]:
df = pd.DataFrame(registros)
df = df.drop_duplicates(subset="id")
df = df[df["nota_promedio"].notna() & (df["nota_promedio"] > 0)]
df = df[df["duracion_min"].fillna(0) > 0]
df = df.reset_index(drop=True)

paths.DATA_RAW.mkdir(parents=True, exist_ok=True)
df.to_csv(paths.RAW_MOVIES_CSV, index=False, encoding="utf-8-sig")

print(f"Dataset final: {df.shape[0]} peliculas x {df.shape[1]} columnas")
print(f"Guardado en: {paths.RAW_MOVIES_CSV.relative_to(paths.PROJECT_ROOT)}")


Dataset final: 3000 peliculas x 25 columnas
Guardado en: data\raw\peliculas_raw.csv


## 5. Verificacion rapida


In [6]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     3000 non-null   int64  
 1   titulo                 3000 non-null   str    
 2   titulo_original        3000 non-null   str    
 3   idioma_original        3000 non-null   str    
 4   fecha_estreno          3000 non-null   str    
 5   anio_estreno           3000 non-null   int64  
 6   mes_estreno            3000 non-null   int64  
 7   estado                 3000 non-null   str    
 8   presupuesto            3000 non-null   int64  
 9   recaudacion            3000 non-null   int64  
 10  duracion_min           3000 non-null   int64  
 11  popularidad            3000 non-null   float64
 12  adultos                3000 non-null   bool   
 13  generos                3000 non-null   str    
 14  genero_principal       3000 non-null   str    
 15  num_generos    

In [7]:
df[["nota_promedio", "cantidad_votos", "presupuesto", "recaudacion", "duracion_min", "popularidad"]].describe()


,nota_promedio,cantidad_votos,presupuesto,recaudacion,duracion_min,popularidad
count,3000.000000,3000.000000,3.000000e+03,3.000000e+03,3000.000000,3000.000000
mean,6.904329,5592.351667,5.388164e+07,1.911743e+08,113.235667,16.016027
std,0.717532,4851.182807,5.971068e+07,2.619364e+08,21.327519,17.232310
min,2.935000,1859.000000,0.000000e+00,0.000000e+00,8.000000,0.020200
25%,6.400000,2533.750000,1.200000e+07,3.735907e+07,98.000000,9.633775
50%,6.902000,3813.000000,3.400000e+07,1.049455e+08,110.000000,12.972250
75%,7.414000,6637.250000,7.500000e+07,2.335164e+08,125.000000,18.207375
max,8.871000,41245.000000,6.588000e+08,2.923706e+09,242.000000,686.080900


In [8]:
df.head()


,id,titulo,titulo_original,idioma_original,fecha_estreno,anio_estreno,mes_estreno,estado,presupuesto,recaudacion,duracion_min,popularidad,adultos,generos,genero_principal,num_generos,sinopsis_es,sinopsis_en,companias_productoras,paises_produccion,reparto_principal,director,keywords,nota_promedio,cantidad_votos
0,157336,Interstellar,Interstellar,en,2014-11-05,2014,11,Released,165000000,746606706,169,71.9338,False,"[""Aventura"", ""Drama"", ""Ciencia ficción""]",Aventura,3,Un grupo de exploradores hacen uso de un aguje...,The adventures of a group of explorers who mak...,"[""Legendary Pictures"", ""Syncopy"", ""Lynda Obst ...","[""United Kingdom"", ""United States of America""]","[""Matthew McConaughey"", ""Anne Hathaway"", ""Mich...",Christopher Nolan,"[""spacecraft"", ""race against time"", ""artificia...",8.488,41245
1,27205,Origen,Inception,en,2010-07-15,2010,7,Released,160000000,839030630,148,59.8123,False,"[""Acción"", ""Ciencia ficción"", ""Aventura""]",Acción,3,"Dom Cobb es un ladrón hábil, el mejor de todos...","Cobb, a skilled thief who commits corporate es...","[""Warner Bros. Pictures""]","[""United Kingdom"", ""United States of America""]","[""Leonardo DiCaprio"", ""Joseph Gordon-Levitt"", ...",Christopher Nolan,"[""mission"", ""dreams"", ""kidnapping"", ""spy"", ""al...",8.374,40255
2,24428,Vengadores,The Avengers,en,2012-04-25,2012,4,Released,220000000,1518815515,142,63.6113,False,"[""Ciencia ficción"", ""Acción"", ""Aventura""]",Ciencia ficción,3,Cuando un enemigo inesperado surge como una gr...,When an unexpected enemy emerges and threatens...,"[""Marvel Studios""]","[""United States of America""]","[""Robert Downey Jr."", ""Chris Evans"", ""Mark Ruf...",Joss Whedon,"[""new york city"", ""superhero"", ""shield"", ""base...",8.077,39815
3,155,El caballero oscuro,The Dark Knight,en,2008-07-16,2008,7,Released,185000000,1004558444,152,63.7889,False,"[""Acción"", ""Suspense"", ""Crimen""]",Acción,3,Batman/Bruce Wayne regresa para continuar su g...,Batman raises the stakes in his war on crime. ...,"[""Warner Bros. Pictures"", ""Legendary Pictures""...","[""United Kingdom"", ""United States of America""]","[""Christian Bale"", ""Heath Ledger"", ""Aaron Eckh...",Christopher Nolan,"[""sadism"", ""chaos"", ""secret identity"", ""crime ...",8.535,36787
4,19995,Avatar,Avatar,en,2009-12-16,2009,12,Released,237000000,2923706026,161,46.7791,False,"[""Ciencia ficción"", ""Acción"", ""Aventura""]",Ciencia ficción,3,"Año 2154. Jake Sully, un exmarine en silla de ...","In the 22nd century, a paraplegic Marine is di...","[""Dune Entertainment"", ""Lightstorm Entertainme...","[""United States of America"", ""United Kingdom""]","[""Sam Worthington"", ""Zoe Saldaña"", ""Sigourney ...",James Cameron,"[""paraplegic"", ""attachment to nature"", ""cultur...",7.609,34718
